In [58]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/satyakidas07/employee-attendance-and-login-logout-data-bangalore/employee_attendance_bangalore_q1_2026.csv


# Employee attendance and login/logout data in bangalore: Exploratory Data Analysis

**Dataset:** `Employee-attendance-and-login-logout-data-bangalore.csv` (55,374 rows × 22 columns)
**Workflow followed:** *Exploratory Data Analysis — A Standard 23-Step Checklist for Any Dataset*
(Classroom Computer Institute — Data Analytics)

This notebook walks through **all 23 steps** of the checklist, in order, across four phases:

| Phase | Steps | Goal |
|---|---|---|
| Phase 1 — Inspect | 1–8 | Understand the shape, structure, and quality of the raw data |
| Phase 2 — Clean & Prepare | 9–16 | Organize columns, clean values, export a final clean dataset |
| Phase 3 — Analyze | 17–19 | Explore relationships and statistically test them |
| Phase 4 — Report | 20-23  | Define KPIs and charts for the final dashboard |

Each step below has its own markdown explanation followed by the code that performs it.

In [59]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

RAW_PATH = "/kaggle/input/datasets/satyakidas07/employee-attendance-and-login-logout-data-bangalore/employee_attendance_bangalore_q1_2026.csv"
df = pd.read_csv(RAW_PATH)
display(df.head())

,attendance_id,employee_id,employee_name,gender,department,designation,employment_type,office_location,date_of_joining,attendance_date,shift_type,attendance_status,work_mode,login_timestamp,logout_timestamp,total_hours_worked,break_duration_mins,net_productive_hours,late_arrival_mins,early_exit_mins,overtime_hours,leave_type
0,ATT0000001,VL1000,Arjun Verma,Male,Engineering,Software Engineer Intern,Intern,Koramangala HQ,2025-06-26,2026-01-02,General (09:30-18:30),Present,Work From Office,2026-01-02 09:42:03,2026-01-02 18:34:12,8.87,86,7.44,12,0,0.00,Not Applicable
1,ATT0000002,VL1001,Rekha Chatterjee,Female,Operations,Operations Manager,Full-Time,Koramangala HQ,2024-02-16,2026-01-02,General (09:30-18:30),Present,Work From Home,2026-01-02 09:09:11,2026-01-02 18:25:11,9.27,67,8.15,0,6,0.00,Not Applicable
2,ATT0000003,VL1002,Tanmay Bhardwaj,Male,Operations,Operations Analyst,Full-Time,Whitefield Tech Park,2020-10-13,2026-01-02,General (09:30-18:30),Present,Work From Home,2026-01-02 09:45:59,2026-01-02 20:17:31,10.53,57,9.58,15,0,0.58,Not Applicable
3,ATT0000004,VL1003,Aditya Bose,Female,Customer Success,Customer Success Executive,Full-Time,Koramangala HQ,2022-01-10,2026-01-02,Late/US Overlap (14:00-23:00),Present,Work From Office,2026-01-02 13:49:26,2026-01-02 22:24:51,8.59,76,7.32,0,36,0.00,Not Applicable
4,ATT0000005,VL1004,Uday Acharya,Male,Human Resources,Talent Acquisition Partner,Full-Time,Electronic City Campus,2024-09-09,2026-01-02,Flexible (10:30-19:30),Present,Work From Office,2026-01-02 10:51:02,2026-01-02 20:33:39,9.71,61,8.69,20,0,0.00,Not Applicable


## Phase 1 — Inspect the Raw Data (Steps 1–8)

Understand the shape, structure, and quality of the raw data.

### Step 1 — Total Number of Rows
Check the row count to understand the dataset's size (`df.shape[0]`).

In [60]:
n_rows = df.shape[0]
print(f"Total rows: {n_rows}")

Total rows: 55374


### Step 2 — Total Number of Columns
Check the column count (`df.shape[1]`).

In [61]:
n_cols = df.shape[1]
print(f"Total columns: {n_cols}")

Total columns: 22


### Step 3 — Understanding of Each Column
Go through every column and note what it represents, its expected values, and how it relates to the problem.

| Column | Represents | Expected values |
|---|---|---|
| attendance_id | Unique identifier for each daily attendance entry | `ATTxxxxxxx` (e.g., `ATT0000001` to `ATT0055374`) |
| employee_id | Unique identifier for each employee | `VLxxxx` (980 unique IDs, e.g., `VL1000` to `VL1979`) |
| employee_name | Full name of the employee | Free text (e.g., Arjun Verma, Rekha Chatterjee) |
| gender | Gender identity of the employee | Male, Female, Prefer Not to Say |
| department | Business department/unit | 10 unique departments (Engineering, Sales, Operations, Data Science, etc.) |
| designation | Job role or title | 48 unique titles (e.g., Software Engineer, Operations Analyst) |
| employment_type | Employment contract type | Full-Time, Contract, Intern, Part-Time |
| office_location | Assigned physical office campus | 5 Bangalore locations (Koramangala HQ, Whitefield Tech Park, HSR Layout Hub, Indiranagar Annexe, Electronic City Campus) |
| date_of_joining | Official employment start date | Date string format `YYYY-MM-DD` |
| attendance_date | Date of the attendance record | Date string format `YYYY-MM-DD` (57 working days in Q1 2026) |
| shift_type | Scheduled work shift and hours | 5 shift types (e.g., General (09:30-18:30), Flexible (10:30-19:30), Mid (11:00-20:00), Late/US Overlap (14:00-23:00), Early (08:00-17:00)) |
| attendance_status | Operational status for the day | Present, Half Day, On Leave |
| work_mode | Work location status | Work From Office, Work From Home, Client Site, Not Applicable (when on leave) |
| login_timestamp | System punch-in date and time | Timestamp `YYYY-MM-DD HH:MM:SS`, blank for leave days |
| logout_timestamp | System punch-out date and time | Timestamp `YYYY-MM-DD HH:MM:SS`, blank for leave days |
| total_hours_worked | Calculated gross duration between login and logout | `0.00` to `13.66` hours (`0.0` on leave days) |
| break_duration_mins | Duration spent on official breaks during shift | `0` to `135` minutes |
| net_productive_hours | Active work duration (`total_hours_worked` minus `break_duration_mins`) | `0.00` to `12.86` hours |
| late_arrival_mins | Delay in login past shift start time | `0` to `218` minutes |
| early_exit_mins | Departure time before shift end time | `0` to `391` minutes |
| overtime_hours | Hours worked beyond standard shift duration | `0.00` to `3.86` hours |
| leave_type | Classification of leave taken | Not Applicable, Sick Leave, Casual Leave, Earned Leave, Loss of Pay, Bereavement Leave, Marriage Leave, Work From Home Comp-Off, Parental Leave |

In [62]:
# Create a summary DataFrame describing structure of df
info_df = pd.DataFrame({
    "dtype": df.dtypes.astype(str),       # Data type of each column
    "n_unique": df.nunique(),             # Number of unique values per column
    "sample_value": df.iloc[0]            # First-row sample value for each column
})

# Optional: reset index to make column names a proper column
info_df = info_df.reset_index().rename(columns={"index": "column"})
info_df

,column,dtype,n_unique,sample_value
0,attendance_id,object,55374,ATT0000001
1,employee_id,object,980,VL1000
2,employee_name,object,980,Arjun Verma
3,gender,object,3,Male
4,department,object,10,Engineering
5,designation,object,48,Software Engineer Intern
6,employment_type,object,4,Intern
7,office_location,object,5,Koramangala HQ
8,date_of_joining,object,804,2025-06-26
9,attendance_date,object,57,2026-01-02


### Step 4 — Trim Extra Spaces
Strip leading/trailing whitespace from string columns and column headers — hidden spaces silently
break groupby, filtering, and joins. We first **detect** which columns are affected before fixing them
(the actual fix happens formally in Step 16, but we flag it here as required by Step 4).

In [63]:
# 1. Check for stray spaces in column names
bad_headers = [col for col in df.columns if col != col.strip()]
print("Bad headers:", bad_headers)

# 2. Check for stray spaces in text cells
for col in df.select_dtypes(include="object"):
    raw = df[col].astype(str)
    issue_count = (raw != raw.str.strip()).sum()

    if issue_count > 0:
        print(f"Column '{col}' has {issue_count} rows with extra spaces.")

Bad headers: []


### Step 5 — Remove Duplicate Elements
Identify and drop duplicate rows (`df.duplicated()`, `df.drop_duplicates()`).

In [64]:
n_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes}")
df.loc[df.duplicated(keep=False)].sort_values("attendance_id").head(6)

Fully duplicated rows: 0


,attendance_id,employee_id,employee_name,gender,department,designation,employment_type,office_location,date_of_joining,attendance_date,shift_type,attendance_status,work_mode,login_timestamp,logout_timestamp,total_hours_worked,break_duration_mins,net_productive_hours,late_arrival_mins,early_exit_mins,overtime_hours,leave_type


### Step 6 — Check Memory Size
Check memory usage (`df.memory_usage(deep=True)`) — flags if dtypes need downcasting for efficiency.

In [65]:
mem = df.memory_usage(deep=True)
print(mem)
print(f"\nTotal memory: {mem.sum() / 1024**2:.2f} MB")

Index                       132
attendance_id           3267066
employee_id             3045570
employee_name           3442982
gender                  2980742
department              3312109
designation             3830839
employment_type         3198638
office_location         3651166
date_of_joining         3267066
attendance_date         3267066
shift_type              3904830
attendance_status       3104765
work_mode               3564478
login_timestamp         3670572
logout_timestamp        3670572
total_hours_worked       442992
break_duration_mins      442992
net_productive_hours     442992
late_arrival_mins        442992
early_exit_mins          442992
overtime_hours           442992
leave_type              3483852
dtype: int64

Total memory: 54.66 MB


### Step 7 — Check Data Type of Columns
Verify each column's dtype matches what it should be (e.g., dates not stored as text, numbers not
stored as objects).

In [66]:
df.dtypes

attendance_id            object
employee_id              object
employee_name            object
gender                   object
department               object
designation              object
employment_type          object
office_location          object
date_of_joining          object
attendance_date          object
shift_type               object
attendance_status        object
work_mode                object
login_timestamp          object
logout_timestamp         object
total_hours_worked      float64
break_duration_mins       int64
net_productive_hours    float64
late_arrival_mins         int64
early_exit_mins           int64
overtime_hours          float64
leave_type               object
dtype: object

### Step 8 — Check Total Number of Null Values
Count missing values per column (`df.isnull().sum()`) to plan the cleaning strategy.

In [67]:
null_counts = df.isnull().sum().sort_values(ascending=False)
null_pct = (null_counts / len(df) * 100).round(2)
pd.DataFrame({"nulls": null_counts, "pct_missing": null_pct}).loc[null_counts > 0]

,nulls,pct_missing
logout_timestamp,2635,4.76
login_timestamp,2635,4.76


## Phase 2 — Clean & Prepare (Steps 9–16)

Organize columns, clean values, and export a final clean dataset.

### Step 9 — Cleaning ID Prefixes and Enforcing Correct Data Types
* **Issue:** `attendance_id` and `employee_id` contain static string prefixes (`ATT` and `VL`) which can be stripped or converted for cleaner indexing.
* **Resolution:** Use `str.replace()` with a try-except block to safely drop prefixes. Then, explicitly convert numeric, datetime, and categorical columns to their proper data types.

In [68]:
# 1. Strip prefixes and convert attendance_id to integer
try:
    df['attendance_id'] = df['attendance_id'].str.replace("ATT", "", regex=False).astype('int64')
    print("Successfully converted 'attendance_id' to integer.")
except Exception as e:
    print("Error converting attendance_id:", e)

# 2. Strip prefixes and convert employee_id to integer
try:
    df['employee_id'] = df['employee_id'].str.replace("VL", "", regex=False).astype('int64')
    print("Successfully converted 'employee_id' to integer.")
except Exception as e:
    print("Error converting employee_id:", e)

# Verify the data types and preview
print("\nUpdated dtypes for ID columns:")
print(df[['attendance_id', 'employee_id']].dtypes)

display(df[['attendance_id', 'employee_id']].head())

Successfully converted 'attendance_id' to integer.
Successfully converted 'employee_id' to integer.

Updated dtypes for ID columns:
attendance_id    int64
employee_id      int64
dtype: object


,attendance_id,employee_id
0,1,1000
1,2,1001
2,3,1002
3,4,1003
4,5,1004


### Step 10 — Step-by-Step Data Cleaning & Transformation

### Step A: Handling Missing Values in Timestamps (`login_timestamp` & `logout_timestamp`)
* **Issue:** There are 2,635 missing entries in login and logout timestamps.
* **Resolution:** Verify that these missing values correspond strictly to employees marked as **"On Leave"** in `attendance_status`.

In [69]:
# Verify if missing timestamps match 'On Leave' status
missing_timestamps_check = df[df['login_timestamp'].isnull()]['attendance_status'].value_counts()
print("Attendance status for rows with missing timestamps:")
print(missing_timestamps_check)

# For structural integrity, we keep them as NaN since employees on leave do not punch in/out.

Attendance status for rows with missing timestamps:
attendance_status
On Leave    2635
Name: count, dtype: int64


### Step B: Splitting Timestamps into Separate Date and Time Columns
* **Issue:** `login_timestamp` and `logout_timestamp` combine date and time strings together.
* **Resolution:** Convert them to proper pandas datetime format and split them into dedicated `login_date`, `login_time`, `logout_date`, and `logout_time` columns. *italicized text*

In [70]:
# Convert to datetime
df['login_timestamp'] = pd.to_datetime(df['login_timestamp'])
df['logout_timestamp'] = pd.to_datetime(df['logout_timestamp'])
df['attendance_date'] = pd.to_datetime(df['attendance_date']).dt.date
df['date_of_joining'] = pd.to_datetime(df['date_of_joining']).dt.date

# Split into separate Date and Time columns
df['login_date'] = df['login_timestamp'].dt.date
df['login_time'] = df['login_timestamp'].dt.time

df['logout_date'] = df['logout_timestamp'].dt.date
df['logout_time'] = df['logout_timestamp'].dt.time

# Preview split columns
display(df[['login_timestamp', 'login_date', 'login_time', 'logout_timestamp', 'logout_date', 'logout_time']].head(3))

,login_timestamp,login_date,login_time,logout_timestamp,logout_date,logout_time
0,2026-01-02 09:42:03,2026-01-02,09:42:03,2026-01-02 18:34:12,2026-01-02,18:34:12
1,2026-01-02 09:09:11,2026-01-02,09:09:11,2026-01-02 18:25:11,2026-01-02,18:25:11
2,2026-01-02 09:45:59,2026-01-02,09:45:59,2026-01-02 20:17:31,2026-01-02,20:17:31


### Step C: Recalculating Time Metrics (Gross Hours & Net Productive Hours)
* **Issue:** Simple subtraction of timestamps doesn't fully account for proper time structures, break durations, late arrivals, or early exits.
* **Resolution:** Re-calculate gross hours cleanly from timestamps, subtract `break_duration_mins` to compute correct `net_productive_hours`.

In [71]:
# Recalculate gross hours worked (in hours)
df['recalculated_gross_hours'] = (df['logout_timestamp'] - df['login_timestamp']).dt.total_seconds() / 3600.0

# Recalculate net productive hours (Gross hours minus break duration in hours)
df['recalculated_net_productive_hours'] = df['recalculated_gross_hours'] - (df['break_duration_mins'] / 60.0)

# Fill NaN for leave records
df['recalculated_gross_hours'] = df['recalculated_gross_hours'].fillna(0)
df['recalculated_net_productive_hours'] = df['recalculated_net_productive_hours'].fillna(0)

display(df[['total_hours_worked', 'recalculated_gross_hours', 'break_duration_mins', 'net_productive_hours', 'recalculated_net_productive_hours']].head())

,total_hours_worked,recalculated_gross_hours,break_duration_mins,net_productive_hours,recalculated_net_productive_hours
0,8.87,8.869167,86,7.44,7.435833
1,9.27,9.266667,67,8.15,8.150000
2,10.53,10.525556,57,9.58,9.575556
3,8.59,8.590278,76,7.32,7.323611
4,9.71,9.710278,61,8.69,8.693611


### Step 11 — Validating and Converting Column Data Types
Let's inspect columns that are currently loaded as objects (text) but should properly be **datetime**, **numeric**, or **categorical/boolean**.

In [72]:
# Define column type groups for our dataset
should_be_datetime = ["date_of_joining", "attendance_date", "login_timestamp", "logout_timestamp"]
should_be_numeric = ["total_hours_worked", "break_duration_mins", "net_productive_hours", "late_arrival_mins", "early_exit_mins", "overtime_hours"]
should_be_categorical = ["gender", "department", "designation", "employment_type", "office_location", "shift_type", "attendance_status", "work_mode", "leave_type"]

print("Currently wrong dtype (datetime expected):")
print(df[should_be_datetime].dtypes)

print("\nCurrently wrong dtype (numeric expected):")
print(df[should_be_numeric].dtypes)

print("\nCurrently wrong dtype (categorical expected):")
print(df[should_be_categorical].dtypes)

# Execute conversions
for col in should_be_datetime:
    df[col] = pd.to_datetime(df[col], errors='coerce')

for col in should_be_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in should_be_categorical:
    df[col] = df[col].astype('category')

print("\n--- All Data Types Successfully Converted ---")
print(df.dtypes)

Currently wrong dtype (datetime expected):
date_of_joining             object
attendance_date             object
login_timestamp     datetime64[ns]
logout_timestamp    datetime64[ns]
dtype: object

Currently wrong dtype (numeric expected):
total_hours_worked      float64
break_duration_mins       int64
net_productive_hours    float64
late_arrival_mins         int64
early_exit_mins           int64
overtime_hours          float64
dtype: object

Currently wrong dtype (categorical expected):
gender               object
department           object
designation          object
employment_type      object
office_location      object
shift_type           object
attendance_status    object
work_mode            object
leave_type           object
dtype: object

--- All Data Types Successfully Converted ---
attendance_id                                 int64
employee_id                                   int64
employee_name                                object
gender                                

In [73]:
# 1. Ensure main timestamps are datetime
df['login_timestamp'] = pd.to_datetime(df['login_timestamp'])
df['logout_timestamp'] = pd.to_datetime(df['logout_timestamp'])

# 2. Create proper Date and Time columns
df['login_date'] = df['login_timestamp'].dt.date
df['login_time'] = df['login_timestamp'].dt.time

df['logout_date'] = df['logout_timestamp'].dt.date
df['logout_time'] = df['logout_timestamp'].dt.time

# Optional: If you want dates stored as actual datetime objects (instead of python 'object' date types) 
# for seamless plotting or filtering, you can strip the time component like this:
df['login_date'] = pd.to_datetime(df['login_timestamp'].dt.date)
df['logout_date'] = pd.to_datetime(df['logout_timestamp'].dt.date)

# Verify updated types
print("Updated Data Types:")
print(df[['login_timestamp', 'login_date', 'login_time', 'logout_timestamp', 'logout_date', 'logout_time']].dtypes)

display(df[['login_date', 'login_time', 'logout_date', 'logout_time']].head(3))

Updated Data Types:
login_timestamp     datetime64[ns]
login_date          datetime64[ns]
login_time                  object
logout_timestamp    datetime64[ns]
logout_date         datetime64[ns]
logout_time                 object
dtype: object


,login_date,login_time,logout_date,logout_time
0,2026-01-02,09:42:03,2026-01-02,18:34:12
1,2026-01-02,09:09:11,2026-01-02,18:25:11
2,2026-01-02,09:45:59,2026-01-02,20:17:31


### Step 12 — Split Columns: Numerical vs Categorical
Separate columns into numerical and categorical groups — they need different analysis and cleaning
approaches. (Split is based on *intended* type, since several numeric columns are still stored as
text at this point.)

In [74]:
numerical_cols = [
    "total_hours_worked",
    "break_duration_mins",
    "net_productive_hours",
    "late_arrival_mins",
    "early_exit_mins",
    "overtime_hours",
]
categorical_cols = [
    "gender",
    "department",
    "designation",
    "employment_type",
    "office_location",
    "shift_type",
    "attendance_status",
    "work_mode",
    "leave_type",
]
datetime_cols = [
    "date_of_joining",
    "attendance_date",
    "login_timestamp",
    "logout_timestamp",
]
id_cols = ["attendance_id", "employee_id"]

print("Numerical:", numerical_cols)
print("\nCategorical:", categorical_cols)
print("\nDate/Time:", datetime_cols)
print("\nID columns:", id_cols)

Numerical: ['total_hours_worked', 'break_duration_mins', 'net_productive_hours', 'late_arrival_mins', 'early_exit_mins', 'overtime_hours']

Categorical: ['gender', 'department', 'designation', 'employment_type', 'office_location', 'shift_type', 'attendance_status', 'work_mode', 'leave_type']

Date/Time: ['date_of_joining', 'attendance_date', 'login_timestamp', 'logout_timestamp']

ID columns: ['attendance_id', 'employee_id']


### Step 13 — `describe()` of Numerical Columns
Run `df.describe()` on our numerical attendance metrics (`total_hours_worked`, `break_duration_mins`, `net_productive_hours`, `late_arrival_mins`, `early_exit_mins`, and `overtime_hours`) to inspect the count, mean, standard deviation, min, quartiles, and max values.

In [75]:
# Define numerical columns for the employee attendance dataset
numerical_cols = [
    "total_hours_worked",
    "break_duration_mins",
    "net_productive_hours",
    "late_arrival_mins",
    "early_exit_mins",
    "overtime_hours"
]

# Generate statistical summary
summary_stats = df[numerical_cols].describe()
display(summary_stats)

,total_hours_worked,break_duration_mins,net_productive_hours,late_arrival_mins,early_exit_mins,overtime_hours
count,55374.000000,55374.000000,55374.000000,55374.000000,55374.000000,55374.000000
mean,8.724951,59.741955,7.729238,9.858471,20.099198,0.150049
std,2.413434,23.878616,2.223308,19.465159,49.630750,0.367258
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8.470000,47.000000,7.370000,0.000000,0.000000,0.000000
50%,9.280000,62.000000,8.210000,0.000000,0.000000,0.000000
75%,10.000000,76.000000,8.960000,14.000000,16.000000,0.000000
max,13.660000,135.000000,12.860000,218.000000,391.000000,3.860000


### Step 14 — Write a Summary of the Dataset

**Summary:** This is a synthetic, daily attendance dataset simulating an organizational workforce operating across five major office locations in Bangalore — Koramangala HQ, Whitefield Tech Park, HSR Layout Hub, Indiranagar Annexe, and Electronic City Campus. Each of the 55,374 rows represents an individual employee's daily attendance record across Q1 2026 (spanning 57 working days), capturing employee demographics, department/designation structures, work shifts, attendance statuses, work modes, login/logout timestamps, gross and net productive hours, break durations, arrival/exit delays, overtime, and leave classifications.

**Source:** Synthetically generated for EDA-workflow and HR data analysis practice.

**Size:** 55,374 rows × 22 columns, covering 980 unique employees.

**General quality:** Deliberately structured with real-world complexities — it contains missing timestamp entries for days when employees are on leave (2,635 records), prefix-formatted ID strings (`ATT` and `VL`), combined date-time timestamp columns requiring separation, and precise operational metrics tracking shift compliance (late arrivals, early exits, breaks, and overtime).

In [76]:
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"Office locations covered: {sorted(df['office_location'].str.strip().unique())}")
print(f"Attendance date range (raw sample): {df['attendance_date'].min()} .. {df['attendance_date'].max()}")
print(f"Attendance status breakdown:\n{df['attendance_status'].value_counts()}")

Rows: 55,374  |  Columns: 28
Office locations covered: ['Electronic City Campus', 'HSR Layout Hub', 'Indiranagar Annexe', 'Koramangala HQ', 'Whitefield Tech Park']
Attendance date range (raw sample): 2026-01-02 00:00:00 .. 2026-03-31 00:00:00
Attendance status breakdown:
attendance_status
Present     51553
On Leave     2635
Half Day     1186
Name: count, dtype: int64


### Step 15 : Standardizing Categorical Columns and Text Strings
* **Issue:** Inconsistent casing, trailing spaces, or text mismatches in columns like `department`, `designation`, `office_location`, `gender`, and `work_mode`.
* **Resolution:** Clean whitespace and standardize string formats.

In [77]:
categorical_cols = ['gender', 'department', 'designation', 'employment_type', 'office_location', 'shift_type', 'attendance_status', 'work_mode', 'leave_type']

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Specific fix for known acronyms or custom stylings if necessary
print("Unique departments after cleaning:", df['department'].unique())
print("Unique work modes after cleaning:", df['work_mode'].unique())

Unique departments after cleaning: ['Engineering' 'Operations' 'Customer Success' 'Human Resources'
 'Marketing' 'Finance' 'Sales' 'Product Management' 'Data Science'
 'Design']
Unique work modes after cleaning: ['Work From Office' 'Work From Home' 'Client Site' 'Not Applicable']


In [80]:
# Final validation checks
print(f"Total Exact Duplicates: {df.duplicated().sum()}")
print(f"Negative Work Hours Count: {(df['total_hours_worked'] < 0).sum()}")
print(f"Negative Net Productive Hours Count: {(df['net_productive_hours'] < 0).sum()}")

# Check timestamp sequence logic (excluding null/leave rows)
valid_timestamps = df.dropna(subset=['login_timestamp', 'logout_timestamp'])
invalid_logins = (pd.to_datetime(valid_timestamps['login_timestamp']) > pd.to_datetime(valid_timestamps['logout_timestamp'])).sum()
print(f"Logins occurring AFTER logouts: {invalid_logins}")

Total Exact Duplicates: 0
Negative Work Hours Count: 0
Negative Net Productive Hours Count: 0
Logins occurring AFTER logouts: 0


### Step 16 — Convert Into Final `cleaned.csv` File
Export the cleaned, prepared dataset as a single `cleaned.csv` to use as the base for all further
analysis.

In [81]:
CLEANED_PATH = "Employee-attendance-and-login-logout-data-bangalore_cleaned.csv"
df.to_csv(CLEANED_PATH, index=False)
print(f"Saved cleaned dataset -> {CLEANED_PATH}  |  shape={df.shape}")

Saved cleaned dataset -> Employee-attendance-and-login-logout-data-bangalore_cleaned.csv  |  shape=(55374, 28)
